# Fase 3 · Núcleo algorítmico, eficiencia y POO
## Licitaciones públicas de ChileCompra · MCDI500 · Grupo 2

F3 organiza las reglas del preprocesamiento de F2 en un pipeline orientado a objetos, contrasta sus dos salidas con las exportaciones de F2 y mide la eficiencia de dos decisiones: el control de fechas y la lectura del archivo. La unidad de análisis de la brecha es la licitación. La base de ofertas conserva el detalle por oferta e ítem.

**Cómo ejecutar.** Deja el archivo de entrada en `data/raw/04LicitacionesEne.csv` (ver *Obtener los datos* en el `README.md`), ejecuta primero `F2/F2_Preprocesamiento.ipynb` para generar las dos referencias en `data/processed/` y luego ejecuta todas las celdas de este notebook con el kernel **Python (grupo2-mcdi500)**. Los módulos están en `src/`: `funciones_f2_revision.py`, `pruebas_f2.py`, `pipeline_licitaciones.py` y `medicion.py`.

## 0. Verificar el entorno de ejecución

Antes de cargar datos comprobamos que el notebook se ejecuta con el entorno virtual del proyecto (`.venv`, kernel **Python (grupo2-mcdi500)**) y que las librerías que usa esta fase tienen las versiones declaradas en `requirements.txt`. Es la misma verificación de la sección 8 de `F1_Definicion`, reducida a lo que necesita la Fase 3.

Importa porque las mediciones de tiempo y memoria de las secciones siguientes dependen del entorno: si cambia la versión de pandas, pueden cambiar las cifras. La celda se detiene si no hay entorno virtual o si falta una librería, y avisa si alguna versión no coincide con la declarada.

In [1]:
# Verificar el entorno: intérprete, entorno virtual y versiones frente a requirements.txt.
import platform
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

# Paquetes que usa la Fase 3, con el nombre que tienen en requirements.txt.
PAQUETES_F3 = ["pandas", "numpy"]


def leer_versiones_declaradas(ruta_requirements, paquetes):
    """Devuelve {paquete: versión} de requirements.txt para los paquetes pedidos."""
    declaradas = {}
    for linea in ruta_requirements.read_text(encoding="utf-8").splitlines():
        nombre, _, version = linea.strip().partition("==")
        if nombre.lower() in paquetes:
            declaradas[nombre.lower()] = version
    return declaradas


def verificar_entorno(paquetes):
    """Comprueba el entorno virtual y compara versiones instaladas y declaradas.

    Se detiene si no hay entorno virtual o si falta un paquete. Devuelve una
    tabla con la versión declarada e instalada de cada paquete.
    """
    inicio = Path.cwd().resolve()
    raiz = next((p for p in (inicio, *inicio.parents) if (p / "requirements.txt").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("No se encontró requirements.txt: abre el notebook dentro del repositorio.")
    if sys.prefix == sys.base_prefix:
        raise RuntimeError("No se usa un entorno virtual: cambia el kernel a 'Python (grupo2-mcdi500)'.")

    declaradas = leer_versiones_declaradas(raiz / "requirements.txt", paquetes)
    filas = []
    for paquete in paquetes:
        try:
            instalada = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            instalada = None
        filas.append({"Paquete": paquete, "Declarada": declaradas.get(paquete),
                      "Instalada": instalada, "Coincide": instalada == declaradas.get(paquete)})
    tabla = pd.DataFrame(filas)

    faltantes = tabla.loc[tabla["Instalada"].isna(), "Paquete"].tolist()
    if faltantes:
        raise ModuleNotFoundError(f"Faltan paquetes: {faltantes}. Ejecuta: pip install -r requirements.txt")

    interprete = Path(sys.executable)
    if interprete.is_relative_to(raiz):
        interprete = interprete.relative_to(raiz)
    print(f"Intérprete: {interprete} | Python {platform.python_version()} | {platform.system()}")
    print("[OK] Entorno virtual activo")
    return tabla


versiones = verificar_entorno(PAQUETES_F3)
display(versiones)
if not versiones["Coincide"].all():
    print("[AVISO] Hay versiones distintas a las declaradas: las cifras de tiempo pueden variar.")
else:
    print(f"[OK] Las {len(versiones)} versiones coinciden con requirements.txt")

Intérprete: .venv/bin/python | Python 3.12.3 | Darwin
[OK] Entorno virtual activo


,Paquete,Declarada,Instalada,Coincide
0,pandas,3.1.0,3.1.0,True
1,numpy,2.5.4,2.5.4,True


[OK] Las 2 versiones coinciden con requirements.txt


## 1. Configurar el proyecto y ejecutar las pruebas de F2

In [2]:
from io import StringIO

inicio = Path.cwd()
raiz = next((p for p in (inicio, *inicio.parents) if (p / "src/pipeline_licitaciones.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("No se encontró src/pipeline_licitaciones.py: abre el notebook dentro del repositorio.")
sys.path.insert(0, str(raiz))

from src.funciones_f2_revision import FECHAS, MONTOS, MOTIVOS, ORDEN_TAMANO, cargar_csv, es_columna_privada
from src.pipeline_licitaciones import (
    Bitacora, ExtremosLogIQR, construir_pipeline, control_iterativo, control_vectorizado,
)
from src.pruebas_f2 import ejecutar_pruebas_f2

entrada = raiz / "data/raw/04LicitacionesEne.csv"
destino = raiz / "data/processed"
referencias = {
    "ofertas": destino / "04LicitacionesEne_procesado.csv",
    "licitaciones": destino / "04LicitacionesEne_brecha_por_licitacion.csv",
}
for ruta in [entrada, *referencias.values()]:
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta {ruta.relative_to(raiz)}. Verifica la entrada y ejecuta F2 primero.")
ejecutar_pruebas_f2()
print("Pruebas de las reglas de F2 aprobadas.")

Pruebas de las reglas de F2 aprobadas.


## 2. Ejecutar el pipeline de F3 y registrar los pasos

- **Herencia.** `TransformacionFija` y `ExtremosLogIQR` implementan el contrato `Transformador.transformar`. Las reglas fijas reutilizan las funciones ya probadas en F2.
- **Encapsulamiento.** `ExtremosLogIQR.ajustar` guarda los límites en `_limites`. `transformar` exige un ajuste previo y vuelve a usar esos límites.
- **Polimorfismo.** El pipeline recorre una secuencia de `Transformador` y llama `paso.transformar(datos)` sin preguntar de qué clase es cada paso; también lee `paso.nombre`, que la clase base declara como parte del contrato. El IQR comparte esa interfaz, pero el pipeline lo ajusta explícitamente antes de aplicarlo.
- **Observer.** `pipeline.suscribir(bitacora.registrar)` añade un observador que recibe los conteos después de cada paso. El pipeline puede ejecutarse sin observadores.
- **División funcional.** Usamos funciones para las reglas y un recorrido iterativo para encadenarlas. Las tablas tienen una secuencia de pasos fija.

La prueba del detector ajusta con montos entre 10 y 50 y evalúa una oferta de 1.000. Esto verifica que al transformar se van a usar los límites calculados con los datos de ajuste.

In [3]:
ajuste = pd.DataFrame({"MonedaOferta": ["CLP"] * 5, "MontoTotalOferta": [10, 20, 30, 40, 50]})
evaluacion = pd.DataFrame({"MonedaOferta": ["CLP", "USD"], "MontoTotalOferta": [1000, 1000]})
detector = ExtremosLogIQR()
detector.ajustar(ajuste)
bandera = detector.transformar(evaluacion)["EsExtremoLogIQR_CLP"]
pd.testing.assert_series_equal(
    bandera, pd.Series([True, pd.NA], dtype="boolean", name="EsExtremoLogIQR_CLP")
)
print("Se comprobó que la aplicación reutiliza los límites aprendidos.")

pipeline = construir_pipeline()
bitacora = Bitacora()
pipeline.suscribir(bitacora.registrar)
resultado = pipeline.ejecutar(entrada)
display(bitacora.tabla())
comparables = resultado.licitaciones["BrechaMonto"].notna()
print(f"Brecha disponible: {comparables.sum():,} de {len(comparables):,} licitaciones.")
print(f"Coincidencias con el monto estimado: {resultado.licitaciones['CoincideConEstimado'].sum():,}")

Se comprobó que la aplicación reutiliza los límites aprendidos.


,Paso,Filas de entrada,Filas de salida
0,Cargar CSV,45741,45741
1,Marcar duplicados potenciales,45741,45741
2,Convertir fechas y montos,45741,45741
3,Normalizar faltantes,45741,45741
4,Ordenar tamaño de proveedor,45741,45741
5,Calcular brecha por licitación,45741,2292
6,Marcar valores extremos CLP,45741,45741
7,Validar ofertas,45741,45741
8,Validar licitaciones,2292,2292


Brecha disponible: 2,257 de 2,292 licitaciones.
Coincidencias con el monto estimado: 1,254


## 3. Comprobar equivalencia con las dos salidas reales de F2

Comparamos cada tabla de F3 con su CSV de referencia generado por F2. Para ambos lados leemos el formato CSV como texto. Así comparamos el orden, las columnas y todos los valores exportados con la misma representación. El CSV no preserva el tipo categórico de pandas. El pipeline comprueba ese tipo en memoria y el diccionario de F3 documenta el orden para recuperarlo.

In [4]:
tablas = {"ofertas": resultado.ofertas, "licitaciones": resultado.licitaciones}
for nombre, tabla in tablas.items():
    referencia = pd.read_csv(
        referencias[nombre], sep=";", encoding="utf-8-sig", dtype="string", keep_default_na=False
    )
    exportada = pd.read_csv(
        StringIO(tabla.to_csv(sep=";", index=False)), sep=";", dtype="string", keep_default_na=False
    )
    pd.testing.assert_frame_equal(referencia, exportada, check_exact=True)
    print(f"Equivalencia F2/F3 en {nombre}: {len(tabla):,} filas y {len(tabla.columns)} columnas.")

Equivalencia F2/F3 en ofertas: 45,741 filas y 17 columnas.
Equivalencia F2/F3 en licitaciones: 2,292 filas y 24 columnas.


## 4. Medir la eficiencia

Las mediciones usan el módulo `src/medicion.py`, que guarda el menor de varios tiempos con `timeit`, mide la memoria pico con `tracemalloc` en una ejecución separada y exige que todas las versiones entreguen exactamente el mismo resultado. La cifra de `tracemalloc` cuenta las asignaciones rastreadas por Python; puede omitir memoria nativa y no representa la memoria total del proceso. Los tiempos dependen del equipo: lo que se conserva entre equipos es la proporción entre versiones.

### 4.1 Verificar la herramienta de medición

Las mediciones de esta fase usan el módulo `src/medicion.py`. Sus funciones son `medir_tiempo()`, que guarda el menor de varios tiempos con `timeit`, `medir_memoria()`, que registra el pico de memoria con `tracemalloc`, y `comparar()`, que mide varias versiones y exige que todas entreguen el mismo resultado. Antes de usarlo sobre los datos comprobamos que funciona en tres escenarios:

- **Caso normal.** Dos formas de sumar los números de 1 a n deben dar el mismo resultado y una fila por versión.
- **Caso límite.** Una sola repetición es válida, y medir la memoria no debe alterar el resultado de la función medida.
- **Excepciones.** Dos versiones con resultados distintos, una sola versión o cero repeticiones deben detenerse con un error claro. Si alguno de esos casos no fallara, la celda se detiene.

In [5]:
# Verificar src/medicion.py en casos normal, límite y de excepción.
from src.medicion import comparar, medir_memoria, medir_tiempo

# Caso normal: dos formas de sumar 1..n dan lo mismo y se obtiene una fila por versión.
n = 10_000
prueba = comparar({"Bucle": lambda: sum(range(n + 1)),
                   "Fórmula": lambda: n * (n + 1) // 2}, repeticiones=3)
assert list(prueba["Implementación"]) == ["Bucle", "Fórmula"]
assert (prueba[["Tiempo mínimo (ms)", "Memoria pico (MB)"]] >= 0).all().all()
print("[OK] Caso normal: dos versiones equivalentes, una fila por versión")

# Caso límite: una sola repetición es válida y medir la memoria no altera el resultado.
assert medir_tiempo(lambda: None, repeticiones=1) >= 0
assert medir_memoria(lambda: 42)[0] == 42
print("[OK] Caso límite: una repetición y resultado intacto")

# Excepciones: cada caso debe fallar con el error indicado.
casos_error = {
    "Resultados distintos": (lambda: comparar({"a": lambda: 1, "b": lambda: 2}), AssertionError),
    "Una sola versión": (lambda: comparar({"a": lambda: 1}), ValueError),
    "Cero repeticiones": (lambda: medir_tiempo(lambda: 1, repeticiones=0), ValueError),
}
for caso, (llamada, error_esperado) in casos_error.items():
    try:
        llamada()
    except error_esperado as error:
        print(f"[OK] Excepción esperada · {caso}: {type(error).__name__} ({error})")
    else:
        raise AssertionError(f"{caso}: se esperaba {error_esperado.__name__} y no hubo error.")

[OK] Caso normal: dos versiones equivalentes, una fila por versión
[OK] Caso límite: una repetición y resultado intacto
[OK] Excepción esperada · Resultados distintos: AssertionError (b entrega un resultado distinto al de la referencia.)
[OK] Excepción esperada · Una sola versión: ValueError (Se necesitan al menos dos implementaciones para comparar.)
[OK] Excepción esperada · Cero repeticiones: ValueError (Se necesita al menos una repetición.)


### 4.2 Control temporal: versión iterativa frente a vectorizada

El control marca un cierre anterior a la publicación o una adjudicación anterior al cierre. Una comparación con una fecha faltante no activa la alerta. Primero comprobamos ambas versiones con casos conocidos; luego `comparar()` mide tiempo y memoria sobre la base completa y exige que las dos marquen exactamente las mismas filas.

In [6]:
casos = pd.DataFrame([
    ["2025-01-01", "2025-01-03", "2025-01-05"],
    ["2025-01-03", "2025-01-02", None],
    [None, "2025-01-05", "2025-01-04"],
    ["2025-01-01", None, "2025-01-05"],
], columns=FECHAS).apply(pd.to_datetime)
esperado = pd.Series([False, True, True, False])
pd.testing.assert_series_equal(control_iterativo(casos), esperado)
pd.testing.assert_series_equal(control_vectorizado(casos), esperado)

ofertas = resultado.ofertas
comparacion_control = comparar({
    "Iterativa": lambda: control_iterativo(ofertas),
    "Vectorizada": lambda: control_vectorizado(ofertas),
})
display(comparacion_control)

cifras = comparacion_control.set_index("Implementación")
razon_tiempo = cifras.loc["Iterativa", "Veces el tiempo de la más rápida"]
razon_memoria = cifras.loc["Iterativa", "Memoria pico (MB)"] / cifras.loc["Vectorizada", "Memoria pico (MB)"]
alertas = control_vectorizado(ofertas)
print(f"Sobre {len(ofertas):,} filas, la versión iterativa tarda {razon_tiempo:,.0f} veces más "
      f"y usa {razon_memoria:,.0f} veces más memoria pico que la vectorizada.")
print(f"Alertas temporales en la base: {int(alertas.sum()):,}. Las dos versiones marcan las mismas filas.")

,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,Iterativa,80.927,4.004,556.523,7
1,Vectorizada,0.145,0.224,1.000,7


Sobre 45,741 filas, la versión iterativa tarda 557 veces más y usa 18 veces más memoria pico que la vectorizada.
Alertas temporales en la base: 0. Las dos versiones marcan las mismas filas.


**Interpretación.** La versión vectorizada es mejor en las dos dimensiones: en esta ejecución fue unas 560 veces más rápida y usó unas 18 veces menos memoria pico, así que no hay que elegir entre tiempo y memoria. Las dos versiones tienen complejidad temporal y espacial **O(n)**: ambas recorren cada fila una vez y guardan una marca por fila. La diferencia está en el costo de cada paso.

- **Tiempo.** La versión iterativa evalúa las comparaciones en Python, fila por fila. La vectorizada compara columnas completas en el código compilado de pandas.
- **Memoria.** La versión iterativa crea una tupla por fila con `itertuples` y acumula las marcas en una lista de objetos de Python antes de convertirla en serie. La vectorizada guarda cada marca en un arreglo booleano de un byte por fila.

En la base real el control no encuentra fechas fuera de secuencia (0 alertas). La comparación mide el costo del control, no un hallazgo sobre los datos.

**Decisión.** Los controles por fila de la Fase 4 se escribirán en forma vectorizada. La versión iterativa se conserva solo como referencia para verificar el resultado.

### 4.3 Leer el archivo: tres estrategias

Cada fase empieza leyendo el archivo de entrada, de unos 57 MB. Hasta esta versión, `cargar_csv()` dejaba que pandas detectara el separador (`sep=None`), lo que obliga a usar el motor de lectura escrito en Python. Comparamos tres estrategias, todas sin los identificadores directos del proveedor:

- **A. Lectura anterior.** Detecta el separador con el motor Python.
- **B. `cargar_csv()` actual.** Declara `sep=";"`, que ya conocemos desde la Fase 1, y usa el motor C de pandas.
- **C. B con `usecols`.** Lee solo las columnas analíticas que el procesado exporta.

Hacemos dos comparaciones porque C entrega menos columnas. En la primera, B debe entregar exactamente el mismo DataFrame que A. En la segunda, C debe coincidir con esas columnas leídas por B. Cada lectura se repite tres veces y se conserva el menor tiempo.

In [7]:
# Comparar tres formas de leer el archivo de entrada: tiempo, memoria y resultado idéntico.
sin_privadas = lambda columna: not es_columna_privada(columna)
COLUMNAS_ANALITICAS = list(MOTIVOS)

# Comparación 1: archivo completo. B (cargador actual) debe entregar lo mismo que A.
lectura_completa = comparar({
    "A. Lectura anterior (sep=None, motor Python)": lambda: pd.read_csv(
        entrada, sep=None, engine="python", encoding="latin-1",
        usecols=sin_privadas, dtype={"NroLicitacion": "string"}),
    "B. cargar_csv() actual (sep=';', motor C)": lambda: cargar_csv(entrada),
}, repeticiones=3)
display(lectura_completa)

# Comparación 2: solo las columnas analíticas. C debe coincidir con esas columnas de B.
lectura_columnas = comparar({
    f"B. Todas las columnas, luego {len(COLUMNAS_ANALITICAS)}": lambda: cargar_csv(entrada)[COLUMNAS_ANALITICAS],
    f"C. usecols, {len(COLUMNAS_ANALITICAS)} columnas": lambda: pd.read_csv(
        entrada, sep=";", encoding="latin-1", usecols=COLUMNAS_ANALITICAS,
        dtype={"NroLicitacion": "string"})[COLUMNAS_ANALITICAS],
}, repeticiones=3)
display(lectura_columnas)

for nombre, tabla in [("B frente a A", lectura_completa), ("C frente a B", lectura_columnas)]:
    veces_tiempo = tabla["Veces el tiempo de la más rápida"].iloc[0]
    veces_memoria = tabla["Memoria pico (MB)"].iloc[0] / tabla["Memoria pico (MB)"].iloc[1]
    print(f"{nombre}: {veces_tiempo:.1f} veces más rápida y {veces_memoria:.1f} veces menos memoria pico, "
          "con el mismo resultado.")

,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,"A. Lectura anterior (sep=None, motor Python)",1259.132,223.130,4.121,3
1,"B. cargar_csv() actual (sep=';', motor C)",305.549,48.654,1.000,3


,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,"B. Todas las columnas, luego 14",300.478,48.653,2.308,3
1,"C. usecols, 14 columnas",130.208,10.916,1.000,3


B frente a A: 4.1 veces más rápida y 4.6 veces menos memoria pico, con el mismo resultado.
C frente a B: 2.3 veces más rápida y 4.5 veces menos memoria pico, con el mismo resultado.


**Interpretación.**

- **A frente a B.** Entregan exactamente el mismo DataFrame: 45.741 filas y 71 columnas (las 72 del archivo sin el identificador que excluye `cargar_csv`), con los mismos tipos y valores. En esta ejecución, B fue unas 4 veces más rápida y usó unas 4,6 veces menos memoria pico. Al detectar el separador, pandas usa el motor Python, que procesa el archivo línea por línea y arma listas intermedias; con el separador declarado, el motor C lee directamente. Como el separador es conocido, detectarlo no aporta nada y cuesta tiempo y memoria.
- **B frente a C.** `usecols` descarta las columnas durante la lectura y no después: con las 14 columnas analíticas, C fue unas 2,3 veces más rápida y usó unas 4,5 veces menos memoria pico. La mejora no es proporcional a las columnas, porque el archivo se sigue recorriendo completo.
- **Complejidad.** Las tres estrategias son **O(n · m)** en tiempo, con n filas y m columnas leídas: todas recorren el archivo una vez. B mejora la constante de cada paso y C reduce m. En memoria, B y C guardan solo el resultado, mientras que A suma las estructuras intermedias del motor Python.

**Decisión.**

1. **La medición cambió el cargador:** `cargar_csv()` pasó de `sep=None` a `sep=";"` (estrategia B). Entrega los mismos datos con menos tiempo y memoria, y la equivalencia de la sección 3 lo confirma sobre las dos salidas de F2.
2. El pipeline sigue leyendo todas las columnas no identificatorias, porque la detección de duplicados potenciales las usa todas. `usecols` queda como práctica recomendada para los análisis de la Fase 4 que necesiten pocas columnas.

## 5. Preparar los insumos para Fase 4

El estudio es descriptivo y usa el conjunto completo. Se entregan las dos unidades verificadas con F2 y un diccionario con tipos y reglas. Los montos de monedas distintas se resumen por separado. La brecha expresa la diferencia entre montos registrados. Para interpretarla como desviación respecto de un presupuesto previo se necesitaría verificar cuándo se registró el estimado.

In [8]:
reglas = {
    **dict.fromkeys(FECHAS, "Conversión de fecha y hora desde el CSV"),
    **dict.fromkeys(MONTOS, "Conversión numérica con coma decimal"),
    "TamanoProveedorOrdinal": "Orden: " + " | ".join(ORDEN_TAMANO),
    "EsDuplicadoPotencial": "Coincidencia exacta entre filas tras excluir identificadores",
    "EsExtremoLogIQR_CLP": "IQR sobre logaritmos de montos positivos en CLP",
    "MontoTotalGanadoras": "Suma de montos ganadores en licitaciones comparables",
    "BrechaMonto": "Monto total ganador menos monto estimado registrado",
    "EstadoBrecha": "Estado de comparación o motivo principal de exclusión",
    "CoincideConEstimado": "Brecha absoluta menor o igual a 0,00000001",
}
diccionario = pd.DataFrame([
    {"Tabla": nombre, "Columna": columna, "Tipo en Python": str(tabla[columna].dtype),
     "Unidad de análisis": "Oferta por ítem" if nombre == "ofertas" else "Licitación",
     "Regla": reglas.get(columna, "Campo de entrada o control de la agregación de F2")}
    for nombre, tabla in tablas.items() for columna in tabla
])
parametros = pd.DataFrame([
    ("Tamaño ordinal", "categorías", " | ".join(ORDEN_TAMANO), "Fijo"),
    ("Brecha", "tolerancia absoluta de coincidencia", 1e-8, "Fijo"),
    ("IQR CLP", "multiplicador", 1.5, "Fijo"),
    *[("IQR CLP", clave, valor, "Ajustado sobre montos positivos en CLP")
      for clave, valor in (resultado.limites_iqr or {}).items()],
], columns=["Paso", "Parámetro", "Valor", "Origen"])

for nombre, tabla in {
    "ofertas_f3.csv": ofertas, "licitaciones_f3.csv": resultado.licitaciones,
    "diccionario_f3.csv": diccionario, "parametros_f3.csv": parametros,
}.items():
    ruta = destino / nombre
    temporal = ruta.with_suffix(".csv.tmp")
    tabla.to_csv(temporal, sep=";", encoding="utf-8-sig", index=False)
    temporal.replace(ruta)
    print(f"{ruta.relative_to(raiz)} · {len(tabla):,} filas")
display(parametros)

data/processed/ofertas_f3.csv · 45,741 filas
data/processed/licitaciones_f3.csv · 2,292 filas
data/processed/diccionario_f3.csv · 41 filas
data/processed/parametros_f3.csv · 8 filas


,Paso,Parámetro,Valor,Origen
0,Tamaño ordinal,categorías,Micro | Pequeña | Mediana | Grande,Fijo
1,Brecha,tolerancia absoluta de coincidencia,0.0,Fijo
2,IQR CLP,multiplicador,1.5,Fijo
3,IQR CLP,ofertas_validas,45603,Ajustado sobre montos positivos en CLP
4,IQR CLP,q1_log,10.683179,Ajustado sobre montos positivos en CLP
5,IQR CLP,q3_log,14.404324,Ajustado sobre montos positivos en CLP
6,IQR CLP,inferior_log,5.101462,Ajustado sobre montos positivos en CLP
7,IQR CLP,superior_log,19.986041,Ajustado sobre montos positivos en CLP


## 6. Verificar las cifras solicitadas por el profesor

Esta revisión usa el CSV local de trabajo. Los conteos se recalculan al ejecutar el notebook. El cuadro de faltantes muestra el tratamiento previsto para las columnas analíticas y los campos con más del 90 % de ausencia. El control de duplicados usa las columnas que quedaron tras excluir los identificadores directos. Conservamos las filas coincidentes y comparamos el resultado al excluir de la comparación las licitaciones con ganadoras potencialmente duplicadas.

La coincidencia se reporta con igualdad numérica exacta y con la tolerancia de F2. La cifra de 1.263 coincidencias indicada por el profesor sirve para contrastar, dado que revisó el CSV original con su propia implementación. El diccionario de la API describe la visibilidad y el monto estimado. Los indicadores derivados de la base y la justificación del estimado ayudan a revisar su origen, pero el momento de registro sigue sin verificarse. La pregunta del proyecto se formula como una comparación descriptiva de los montos registrados.

In [9]:
from src.funciones_f2_revision import (
    convertir_tipos, marcar_duplicados, normalizar_codigo, tratar_faltantes,
)

entrada_auditada = cargar_csv(entrada)
nulos_al_leer = entrada_auditada.isna().sum()
marcadas, duplicados = marcar_duplicados(entrada_auditada)
convertidas, conversiones = convertir_tipos(marcadas)
_, faltantes = tratar_faltantes(convertidas, nulos_al_leer.reindex(convertidas.columns))
faltantes = faltantes.drop(index="EsDuplicadoPotencial")
print(f"Columnas de entrada tras el filtro de identificadores: {entrada_auditada.shape[1]}")
print(f"Filas adicionales idénticas: {int(duplicados['FilasAdicionales'].sum()):,}")
display(conversiones)
display(faltantes.loc[faltantes.index.isin(MOTIVOS) | faltantes["% faltantes"].gt(90)])

brechas = resultado.licitaciones
validas = brechas.loc[brechas["EstadoBrecha"].eq("Calculada")]
sin_duplicadas = validas.loc[~validas["GanadoraDuplicada"]]
print(f"Cobertura por licitación: {len(validas):,} de {len(brechas):,} ({len(validas) / len(brechas):.1%})")
for etiqueta, muestra in [("Todas las comparables", validas), ("Sensibilidad sin ganadoras duplicadas", sin_duplicadas)]:
    print(etiqueta)
    print(f"Licitaciones: {len(muestra):,}")
    print(f"Coincidencias exactas: {muestra['BrechaMonto'].eq(0).sum():,}")
    print(f"Coincidencias con tolerancia F2: {muestra['CoincideConEstimado'].sum():,}")
print(f"Diferencia frente a las 1.263 coincidencias reportadas por el profesor: "
      f"{int(validas['CoincideConEstimado'].sum()) - 1263:+,}")

if "MontoEstimadoVisible" in entrada_auditada:
    visibilidad = normalizar_codigo(entrada_auditada["MontoEstimadoVisible"])
    no_visibles = visibilidad.eq("NO").sum()
    print(f"Ofertas con monto estimado no visible: {no_visibles:,} de {len(ofertas):,} ({no_visibles / len(ofertas):.1%})")
    display(validas.groupby(["MonedaLicitacion", "VisibilidadEstimado"]).agg(
        Licitaciones=("BrechaMonto", "size"),
        Coincidencias=("CoincideConEstimado", "sum"),
        MedianaBrecha=("BrechaMonto", "median"),
    ))

contexto = [col for col in brechas if col.endswith((
    "_Informada", "_MencionaAdjudicacion", "_MencionaPresupuesto",
))]
if contexto:
    display(validas.groupby("CoincideConEstimado")[contexto].sum().T)
print("Valores extremos por IQR logarítmico en CLP:")
display(ofertas["EsExtremoLogIQR_CLP"].value_counts(dropna=False).to_frame("Ofertas"))

Columnas de entrada tras el filtro de identificadores: 71
Filas adicionales idénticas: 22


,Nulos en entrada,Fallos de conversión
Columna,,
FechaPublicacion,0,0
FechaCierre,0,0
FechaAdjudicacion,91,0
MontoNetoOferta,0,0
MontoTotalOferta,0,0
MontoEstimadoLicitacion,0,0


,Nulos al leer,Nulos después de convertir y normalizar,% faltantes,Regla,Motivo
NroLicitacion,0.0,0,0.000000,Conservar faltantes,"Identifica la licitación. Sin código, la fila ..."
MonedaLicitacion,0.0,0,0.000000,Conservar faltantes,"Unidad del estimado. Si falta, no puede verifi..."
MontoEstimadoLicitacion,0.0,0,0.000000,Conservar faltantes,Referencia de comparación. Sin monto válido se...
MontoEstimadoVisible,0.0,0,0.000000,Conservar faltantes,Distingue visibilidad pública del monto; vacío...
FechaPublicacion,0.0,0,0.000000,Conservar faltantes,Permite describir cuándo se publicó. Se descon...
FechaCierre,0.0,0,0.000000,Conservar faltantes,Permite describir el plazo de recepción cuando...
FechaAdjudicacion,91.0,91,0.198946,Conservar faltantes,Permite estudiar fechas de adjudicación observ...
FechaEntregaEnSoporteFisico,45457.0,45457,99.379113,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
FechaEstimadaEvaluacionOfertas,43489.0,43489,95.076627,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
LicitacionBaseTipo,45741.0,45741,100.000000,Excluir del CSV analítico,Sin observaciones y fuera del análisis.


Cobertura por licitación: 2,257 de 2,292 (98.5%)
Todas las comparables
Licitaciones: 2,257
Coincidencias exactas: 1,254
Coincidencias con tolerancia F2: 1,254
Sensibilidad sin ganadoras duplicadas
Licitaciones: 2,253
Coincidencias exactas: 1,254
Coincidencias con tolerancia F2: 1,254
Diferencia frente a las 1.263 coincidencias reportadas por el profesor: -9
Ofertas con monto estimado no visible: 24,895 de 45,741 (54.4%)


Licitaciones  Coincidencias  \
MonedaLicitacion VisibilidadEstimado                                
CLF              NO                             10              0   
                 SI                             10              0   
CLP              NO                           1189            635   
                 SI                           1041            617   
USD              NO                              1              1   
                 SI                              2              1   
UTM              NO                              4              0   

                                      MedianaBrecha  
MonedaLicitacion VisibilidadEstimado                 
CLF              NO                         -114.58  
                 SI                         -21.595  
CLP              NO                             0.0  
                 SI                             0.0  
USD              NO                             0.0  
                 SI                          -0.412  
UTM              NO                          -31.64

CoincideConEstimado,False,True
BaseEstimacionMontoLicitacion_Informada,1003,1254
BaseEstimacionMontoLicitacion_MencionaAdjudicacion,0,0
BaseEstimacionMontoLicitacion_MencionaPresupuesto,666,907
JustificacionMontoEstimado_Informada,313,475
JustificacionMontoEstimado_MencionaAdjudicacion,2,1
JustificacionMontoEstimado_MencionaPresupuesto,105,155


Valores extremos por IQR logarítmico en CLP:


,Ofertas
EsExtremoLogIQR_CLP,
False,44716
True,887
<NA>,138


## 7. Evidencias para el informe técnico

- Incluye el enlace al commit del notebook y del módulo, la bitácora del pipeline y ejemplos de las clases y de la suscripción del observador.
- Reporta las dimensiones y el resultado de la comparación con **cada** salida de F2.
- Incorpora las tablas de tiempo y memoria de las secciones 4.2 y 4.3 y el entorno de la sección 0. Distingue el costo lineal teórico de los tiempos observados y explica cómo la medición de 4.3 cambió el cargador.
- Incluye la verificación de `src/medicion.py` de la sección 4.1 como evidencia de pruebas en casos normal, límite y de excepción.
- Registra los parámetros ajustados del IQR y explica por qué los dos archivos analíticos tienen unidades diferentes.
- Usa las cifras del bloque 6 de esta versión. Las 502 brechas y los 7.635 atípicos citados en la devolución corresponden al método anterior.
- Antes de entregar, comprueba que el repositorio excluya el CSV original, que el README indique cómo obtener la fuente y dónde colocarla, y que el equipo gestione la eliminación de copias con identificadores que sigan en el historial. Mantén locales las columnas de texto libre. Registra los aportes del equipo con sus propios commits.